## 죽지 않는 프로그램: try-except

파이썬에서 에러가 발생할 가능성이 있는 코드는 `try` 블록으로 감싸고, 에러 발생 시 대처 방안을 `except` 블록에 작성한다.

In [1]:
# 0이 들어 있는 리스트 생성
data = [10, 20, 0, 30, 40]

In [2]:

# [Bad] 에러 처리가 없는 코드
# data 리스트에 0이 있으면 ZeroDivisionError가 발생하고 프로그램이 즉시 종료된다.
for x in data:
    print(100 / x)

10.0
5.0


ZeroDivisionError: division by zero

In [3]:
# [Good] 예외 처리를 적용한 코드
for x in data:
    try:
        result = 100 / x
        print(f"결과: {result}")
    except ZeroDivisionError:
        # 에러가 나도 죽지 않고 다음 데이터로 넘어간다.
        print("에러: 0으로 나눌 수 없습니다. 건너뜁니다.")
    except Exception as e:
        # 그 외 예상치 못한 모든 에러를 잡는다.
        print(f"알 수 없는 에러 발생: {e}")

결과: 10.0
결과: 5.0
에러: 0으로 나눌 수 없습니다. 건너뜁니다.
결과: 3.3333333333333335
결과: 2.5


핵심은 **"구체적인 예외를 먼저 잡는 것"** 이다. 

모든 에러를 퉁쳐서 `except:`로 잡는 것은 좋지 않다. 내가 예측한 에러(`ZeroDivisionError`)와 예측하지 못한 에러(`Exception`)를 구분해야 디버깅이 쉬워진다.


### 실전: 파일로 로그 남기기

아래 코드는 화면에는 'INFO' 레벨 이상만 보여주고, 파일에는 'DEBUG' 레벨까지 상세하게 기록하는 설정 예시다. 
이것이 바로 **실무형 로깅**이다.


In [4]:
import logging

# 1. 로거(Logger) 생성 및 레벨 설정
logger = logging.getLogger()
logger.setLevel(logging.DEBUG) # 중요: 루트 로거는 일단 모든 로그(DEBUG 이상)를 다 받는다.

# 2. 포맷터(Formatter) 설정 (날짜-레벨-메시지)
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')

# 3. 핸들러 1: 파일 저장용 (블랙박스) -> DEBUG 레벨까지 다 저장
file_handler = logging.FileHandler("research_log.txt", encoding='utf-8')
file_handler.setLevel(logging.DEBUG) 
file_handler.setFormatter(formatter)

# 4. 핸들러 2: 화면 출력용 (계기판) -> INFO 레벨 이상만 출력
stream_handler = logging.StreamHandler()
stream_handler.setLevel(logging.INFO) # 여기가 핵심! 화면에는 INFO부터만 보인다.
stream_handler.setFormatter(formatter)

# 5. 로거에 핸들러 등록
logger.addHandler(file_handler)
logger.addHandler(stream_handler)

In [5]:
# --- 테스트 함수 ---
def process_data(data_list):
    logging.info("데이터 처리를 시작합니다.") # 화면 O, 파일 O

    success_count = 0
    for idx, item in enumerate(data_list):
        try:
            # 상세한 처리 과정은 DEBUG로 기록 (화면 X, 파일 O)
            logging.debug(f"{idx}번째 데이터 파싱 시도: {item}")

            result = 100 / int(item)

            logging.debug(f" -> 연산 성공: {result}") # 화면 X, 파일 O
            success_count += 1

        except ValueError:
            logging.warning(f"데이터 형식이 잘못됨: {item}") # 화면 O, 파일 O
        except ZeroDivisionError:
            logging.error(f"0으로 나누기 에러: {item}") # 화면 O, 파일 O

    logging.info(f"처리 종료. 성공: {success_count}/{len(data_list)}") # 화면 O, 파일 O

# 실행
raw_data = ["10", "20", "0", "오류", "50"]
process_data(raw_data)

2026-02-24 10:58:47,979 - INFO - 데이터 처리를 시작합니다.
2026-02-24 10:58:47,981 - ERROR - 0으로 나누기 에러: 0
2026-02-24 10:58:47,982 - WARNING - 데이터 형식이 잘못됨: 오류
2026-02-24 10:58:47,982 - INFO - 처리 종료. 성공: 3/5


## TIP: 허락보다 용서가 쉽다. (EAFP vs LBYL)

파이썬에는 독특한 코딩 철학이 있습니다. 바로 **EAFP (Easier to Ask for Forgiveness than Permission)** 입니다.

* **LBYL (Look Before You Leap):** "뛰기 전에 살펴라." 전통적인 C/Java 스타일입니다. 조건을 먼저 꼼꼼히 검사합니다.  

In [6]:
import os 
filename = "sample_data.txt"

if os.path.exists(filename): # 파일이 있는지 확인하고 (허락)
    open(filename)           # 연다
else:
    print("파일 없음")

파일 없음


* **EAFP:** "허락받는 것보다 저지르고 용서비는 게 빠르다." 파이썬 스타일입니다. 일단 지르고 봅니다.  

In [7]:
try:
    open(filename)           # 일단 연다 (저지름)
except FileNotFoundError:    # 에러가 나면 처리한다 (용서)
    print("파일 없음")

파일 없음


왜 파이썬은 EAFP를 선호할까요?  

1.  **가독성:** 정상적인 흐름(Happy Path)이 먼저 눈에 들어옵니다.  
2.  **경쟁 상태(Race Condition) 방지:** LBYL 방식에서는 '파일 확인'과 '파일 열기' 사이의 찰나의 순간에 파일이 삭제될 수 있습니다. EAFP는 그럴 걱정이 없습니다.

"**일단 시도하고(try), 문제가 생기면 수습(except)하세요.**" 이것이 파이썬다운 태도입니다.